# Resilience Patterns

This deep dive extends [notebook 10](/courses/llm-eng/10-model-serving.html) and [notebook 12](/courses/llm-eng/12-ai-infra.html), which establish the serving and infrastructure foundations. Production AI services have failure modes that pure software does not. A microservice downstream from a database can assume that a valid SQL query always returns a valid result in bounded time; an LLM service cannot. Models time out under load, rate limits impose hard ceilings on throughput, context windows overflow on long documents, and — most insidiously — a model can return syntactically valid JSON that is semantically wrong. This notebook builds the full resilience toolkit for a financial AI service: exponential backoff with jitter, circuit breakers, fallback chains, bulkheads, graceful degradation, and context-overflow handling. We assemble all patterns into a `ResilientComplianceService` and run a simulation with probabilistic fault injection to demonstrate the system continuing to serve safe, degraded responses under stress.

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Failure Taxonomy

Not all LLM failures are alike. The resilience pattern we apply depends entirely on which category an error falls into:

<br>

**Transient errors.** Errors that are likely to resolve on their own: HTTP 429 rate-limit responses, HTTP 503 service unavailability, and network timeouts. The defining property is that the exact same request, retried after a short delay, has a reasonable chance of succeeding. The appropriate response is [retry with backoff]{.mark}.

<br>

**Permanent errors.** Errors for which retrying the identical request will always fail: a prompt that exceeds the model's context window, a malformed API request, or invalid credentials. Retrying wastes quota and time. The appropriate response is [fail fast and route to an alternative]{.mark}.

<br>

**Quality degradation.** The request succeeds — the model returns a 200 with well-formed JSON — but the answer is wrong, incomplete, or hallucinated. This is the hardest failure mode to detect because there is no exception to catch. Detection requires application-layer checks: schema validation, confidence scores, or a lightweight judge. The appropriate response is [escalate to a stronger model or flag for human review]{.mark}.

<br>

For a financial AI service, we define the following SLO targets:

| Metric | Target | Notes |
| :-- | :--: | :-- |
| p99 latency | $< 2\,\text{s}$ | End-to-end, including retrieval |
| Error rate | $< 0.1\%$ | Unhandled exceptions reaching the caller |
| Faithfulness | $> 0.85$ | RAGAS faithfulness on a held-out eval set |
| Degraded-safe rate | $100\%$ | Every failure returns a human-reviewable result |

: SLO targets for the compliance review service. {tbl-colwidths="[30,15,55]"}

## Exponential Backoff with Jitter

When multiple clients hit a rate limit simultaneously and all retry on the same schedule, they produce a **thundering herd**: the retries arrive at the server at the same time and immediately trigger another 429. The solution is to add randomness — **jitter** — to the retry delay. With full jitter, the wait time after attempt $n$ is drawn from:

$$t_n = \text{base} \cdot 2^n \cdot U(0, 1)$$

where $U(0, 1)$ is a uniform random variable. This spreads retries across the interval $[0,\, \text{base} \cdot 2^n]$, reducing the probability that many clients retry at exactly the same moment. We cap $t_n$ at a maximum delay to prevent indefinite backoff under sustained overload:

$$t_n = \min\!\left(\text{cap},\; \text{base} \cdot 2^n \cdot U(0, 1)\right).$$

We implement `RetryClient` using `tenacity`, which provides declarative retry semantics and handles the retry loop, exception filtering, and wait strategy for us.

Defining the retry wrapper:

In [ ]:
import time
import random
import logging
from tenacity import (
    retry,
    stop_after_attempt,
    wait_random_exponential,
    retry_if_exception_type,
    before_sleep_log,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger(__name__)


class RateLimitError(Exception):
    """Simulated transient rate-limit error."""


class RetryClient:
    """LLMClient wrapper with exponential backoff + full jitter."""

    def __init__(self, inner: LLMClient, max_attempts: int = 4,
                 base: float = 0.5, cap: float = 10.0):
        self._inner = inner
        self._max_attempts = max_attempts
        self._base = base
        self._cap = cap
        self._attempt_log: list[int] = []

    def complete(self, messages, **kwargs):
        @retry(
            stop=stop_after_attempt(self._max_attempts),           # <1>
            wait=wait_random_exponential(                          # <2>
                multiplier=self._base, max=self._cap
            ),
            retry=retry_if_exception_type(RateLimitError),        # <3>
            before_sleep=before_sleep_log(logger, logging.WARNING),
            reraise=True,
        )
        def _call():
            return self._inner.complete(messages, **kwargs)

        return _call()

1. Stop retrying after `max_attempts` total calls (including the first).
2. `wait_random_exponential` implements full-jitter: the wait before attempt $n$ is sampled from $U(0, \text{base} \cdot 2^n)$, capped at `max` seconds.
3. We only retry `RateLimitError` (transient). Permanent errors like `ContextOverflowError` will propagate immediately.

To demonstrate recovery, we build a mock LLM that fails with a rate-limit error on its first two calls:

In [ ]:
class FlakeyLLM:
    """Stub LLM: raises RateLimitError for the first `fail_count` calls."""

    def __init__(self, fail_count: int = 2, response: str = "COMPLIANT"):
        self._fail_count = fail_count
        self._calls = 0
        self._response = response
        self.model = "mock"

    def complete(self, messages, **kwargs):
        self._calls += 1
        if self._calls <= self._fail_count:
            print(f"  [FlakeyLLM] call {self._calls}: raising RateLimitError")
            raise RateLimitError("429 Too Many Requests")
        print(f"  [FlakeyLLM] call {self._calls}: success")
        return self._response


flakey = FlakeyLLM(fail_count=2)
client = RetryClient(flakey, max_attempts=4, base=0.01, cap=0.1)  # short delays for demo

print("Sending compliance review request...")
result = client.complete([{"role": "user", "content": "Review trade order #TRD-8821."}])
print(f"Result: {result}")

## Circuit Breaker

Retrying a service that is completely down is wasteful: each attempt occupies a thread, consumes timeout budget, and returns an error anyway. The **circuit breaker** pattern short-circuits calls to a failing dependency rather than letting them accumulate. It models the dependency as a state machine with three states:

- **CLOSED** — normal operation; all requests flow through.
- **OPEN** — the dependency is considered down; requests fail immediately without being forwarded.
- **HALF_OPEN** — a cooldown period has elapsed; a single probe request is forwarded to test recovery.

Transitions are governed by a sliding-window failure rate and a cooldown timer:

$$\text{CLOSED} \xrightarrow{\;\text{failure rate} > \theta\;} \text{OPEN} \xrightarrow{\;t > t_{\text{cool}}\;} \text{HALF\_OPEN} \xrightarrow{\;\text{probe success}\;} \text{CLOSED}$$

If the probe in HALF_OPEN fails, the breaker returns immediately to OPEN and resets the cooldown clock.

Implementing the state machine:

In [ ]:
from enum import Enum, auto
from collections import deque


class CBState(Enum):
    CLOSED    = auto()
    OPEN      = auto()
    HALF_OPEN = auto()


class CircuitBreakerOpen(Exception):
    """Raised when the circuit is OPEN and a request is rejected."""


class CircuitBreaker:
    """Sliding-window circuit breaker wrapping any object with a .complete() method."""

    def __init__(self, inner, window: int = 5,
                 failure_threshold: float = 0.6,
                 cooldown: float = 5.0):
        self._inner = inner
        self._window = window                        # <1>
        self._threshold = failure_threshold          # <2>
        self._cooldown = cooldown
        self._state = CBState.CLOSED
        self._results: deque[bool] = deque(maxlen=window)  # True=success
        self._opened_at: float | None = None

    @property
    def state(self) -> CBState:
        return self._state

    def _failure_rate(self) -> float:
        if not self._results:
            return 0.0
        return 1.0 - (sum(self._results) / len(self._results))

    def _try_reset(self) -> None:
        """Check whether cooldown has elapsed and transition to HALF_OPEN."""
        if self._state == CBState.OPEN:
            elapsed = time.time() - self._opened_at
            if elapsed >= self._cooldown:
                self._state = CBState.HALF_OPEN
                print(f"  [CB] → HALF_OPEN after {elapsed:.1f}s cooldown")

    def complete(self, messages, **kwargs):
        self._try_reset()

        if self._state == CBState.OPEN:              # <3>
            raise CircuitBreakerOpen("Circuit is OPEN — request rejected")

        try:
            result = self._inner.complete(messages, **kwargs)
            self._results.append(True)
            if self._state == CBState.HALF_OPEN:     # <4>
                self._state = CBState.CLOSED
                self._results.clear()
                print(f"  [CB] → CLOSED on probe success")
            return result

        except Exception as exc:
            self._results.append(False)
            rate = self._failure_rate()
            if self._state == CBState.HALF_OPEN:     # <5>
                self._state = CBState.OPEN
                self._opened_at = time.time()
                print(f"  [CB] → OPEN (probe failed)")
            elif rate >= self._threshold and len(self._results) == self._window:
                self._state = CBState.OPEN
                self._opened_at = time.time()
                print(f"  [CB] → OPEN (failure rate={rate:.0%})")
            raise exc

1. We track the last `window` outcomes; older results fall off automatically thanks to `deque(maxlen=window)`.
2. If the failure rate over the window exceeds `failure_threshold`, the breaker opens.
3. In OPEN state, all calls fail immediately — no forwarding occurs.
4. A successful probe in HALF_OPEN closes the breaker and clears the results window so stale failures don't immediately reopen it.
5. A failed probe in HALF_OPEN reopens the breaker and resets the cooldown clock.

Simulating a burst of failures followed by recovery:

In [ ]:
class CountingLLM:
    """Stub LLM: fails for call indices in `fail_on`."""

    def __init__(self, fail_on: set[int]):
        self._fail_on = fail_on
        self._calls = 0
        self.model = "mock"

    def complete(self, messages, **kwargs):
        self._calls += 1
        if self._calls in self._fail_on:
            raise RuntimeError(f"Simulated failure on call {self._calls}")
        return "COMPLIANT"


# Calls 1-4 fail → window of 5 filled with 4 failures → OPEN
mock_llm = CountingLLM(fail_on={1, 2, 3, 4})
cb = CircuitBreaker(mock_llm, window=5, failure_threshold=0.6, cooldown=0.1)

for i in range(10):
    try:
        resp = cb.complete([{"role": "user", "content": f"Review order #{i}"}])
        print(f"  call {i+1}: OK → state={cb.state.name}")
    except CircuitBreakerOpen:
        print(f"  call {i+1}: REJECTED (circuit open) → state={cb.state.name}")
    except RuntimeError as e:
        print(f"  call {i+1}: ERROR ({e}) → state={cb.state.name}")
    if i == 5:                        # pause to let cooldown elapse
        time.sleep(0.15)

:::{.callout-caution}
The circuit breaker transitions to OPEN only after the window is fully populated. Early in the service's lifetime — when the results deque has fewer than `window` entries — no threshold check fires even if every call fails. Set `window` small (3–5) for fast-failure services, and ensure the service is pre-warmed or that startup errors are excluded from the window.

:::

## Fallback Chains

When a primary model is unavailable — circuit open, quota exhausted, or context overflow — we want to serve a degraded but useful answer rather than an error. A **fallback chain** tries each option in descending order of capability:

1. `gpt-4o` — highest quality, highest cost, primary path.
2. `gpt-4o-mini` — lower cost, slightly lower quality, acceptable for most queries.
3. Rule-based extractor — a regex scan of the input for compliance keywords; no LLM call, zero cost, zero latency, but no reasoning.

The cost–quality tradeoff is explicit: as we descend the chain, we trade answer quality for availability. The invariant we preserve is that the service always returns *something* actionable rather than an exception.

Implementing the rule-based fallback and the chain:

In [ ]:
import re
from dataclasses import dataclass, field


COMPLIANCE_KEYWORDS = [
    r"\bsanction", r"\bwatchlist", r"\bAML", r"\bKYC",
    r"\bfraud", r"\bunauthorized", r"\brestricted",
    r"\bembargo", r"\bmoney.laundering",
]
_PATTERN = re.compile("|".join(COMPLIANCE_KEYWORDS), re.IGNORECASE)


@dataclass
class ReviewResult:
    verdict: str          # "COMPLIANT" | "NON_COMPLIANT" | "REVIEW_REQUIRED"
    explanation: str
    source: str           # which chain link produced the result
    cost_usd: float = 0.0


def rule_based_review(text: str) -> ReviewResult:
    """Regex compliance keyword extractor — zero-cost fallback."""
    hits = _PATTERN.findall(text)
    if hits:
        return ReviewResult(
            verdict="REVIEW_REQUIRED",
            explanation=f"Keyword matches: {list(set(h.upper() for h in hits))}. Human review required.",
            source="rule-based",
        )
    return ReviewResult(
        verdict="COMPLIANT",
        explanation="No compliance keywords detected by rule-based scan.",
        source="rule-based",
    )


class FallbackChain:
    """Tries each link in order; returns the first successful result."""

    def __init__(self, links: list):
        self._links = links   # list of (name, callable) tuples

    def review(self, text: str) -> ReviewResult:
        for name, fn in self._links:
            try:
                result = fn(text)          # <1>
                print(f"  [FallbackChain] succeeded with '{name}'")
                return result
            except Exception as e:
                print(f"  [FallbackChain] '{name}' failed: {e}")
        # The rule-based link never raises, so this is unreachable in practice
        raise RuntimeError("All fallback links exhausted")  # <2>

1. Each link is a callable `fn(text) -> ReviewResult`. The chain is link-agnostic: LLM clients and rule-based functions share the same protocol.
2. This should never execute if the final link is rule-based (which always succeeds). It exists as a defensive assertion.

Wiring up the chain with a broken primary and a working mini model:

In [ ]:
SYSTEM_PROMPT = (
    "You are a compliance officer. Review the trade description and respond with "
    "exactly one word: COMPLIANT or NON_COMPLIANT."
)


def llm_review(client: LLMClient, text: str) -> ReviewResult:
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text},
    ]
    verdict = client.complete(msgs).strip().upper()
    return ReviewResult(
        verdict=verdict,
        explanation=f"{client.model} classification.",
        source=client.model,
        cost_usd=client.total_cost,
    )


# Primary: always raises (simulates outage)
class BrokenLLM:
    model = "gpt-4o"
    def complete(self, *a, **kw):
        raise CircuitBreakerOpen("Circuit OPEN for gpt-4o")


broken_4o  = BrokenLLM()
mini_client = LLMClient(model="gpt-4o-mini")

chain = FallbackChain([
    ("gpt-4o",      lambda t: llm_review(broken_4o,   t)),
    ("gpt-4o-mini", lambda t: llm_review(mini_client, t)),
    ("rule-based",  lambda t: rule_based_review(t)),
])

trade_desc = (
    "Sell 5000 shares of XYZ Corp for client account #8821. "
    "Client is on internal watchlist for AML review."
)

result = chain.review(trade_desc)
print(f"\nVerdict : {result.verdict}")
print(f"Source  : {result.source}")
print(f"Explanation: {result.explanation}")
print(f"Cost    : ${result.cost_usd:.6f}")

## Bulkheads

A **bulkhead** partitions capacity so that one consumer cannot starve another. In ship design, bulkheads are watertight compartments that prevent a single breach from flooding the whole hull. In an AI service, the analogous failure is a burst of requests from tenant A filling all available concurrency slots, blocking tenant B's time-sensitive margin-call processor.

We implement per-tenant concurrency limits using `asyncio.Semaphore`. Each tenant gets a semaphore capped at `max_concurrent` slots. Requests that would exceed the limit raise `BulkheadFull` immediately rather than queuing indefinitely — a pattern called **fail fast** that preserves predictable latency.

Implementing the bulkhead executor:

In [ ]:
import asyncio


class BulkheadFull(Exception):
    """Raised when no concurrency slots are available for a tenant."""


class BulkheadExecutor:
    """Per-tenant async concurrency limiter."""

    def __init__(self, max_concurrent: int = 2):
        self._max = max_concurrent
        self._semaphores: dict[str, asyncio.Semaphore] = {}
        self._active: dict[str, int] = {}

    def _get_sem(self, tenant: str) -> asyncio.Semaphore:
        if tenant not in self._semaphores:
            self._semaphores[tenant] = asyncio.Semaphore(self._max)  # <1>
            self._active[tenant] = 0
        return self._semaphores[tenant]

    async def execute(self, tenant: str, coro):
        sem = self._get_sem(tenant)
        if sem._value == 0:                  # <2>
            raise BulkheadFull(f"Tenant '{tenant}' bulkhead full ({self._max} slots used)")
        async with sem:
            self._active[tenant] += 1
            try:
                return await coro
            finally:
                self._active[tenant] -= 1

1. Each tenant gets an independent semaphore. Tenant A's semaphore exhaustion has no effect on tenant B's semaphore.
2. We check whether the semaphore is saturated *before* acquiring it. This gives fail-fast semantics: rather than queuing behind the semaphore, we immediately raise `BulkheadFull`.

Demonstrating tenant isolation under a burst from tenant A:

In [ ]:
async def mock_llm_call(tenant: str, req_id: int, latency: float = 0.05) -> str:
    """Simulate a slow LLM request."""
    await asyncio.sleep(latency)
    return f"{tenant}:{req_id}:COMPLIANT"


async def run_bulkhead_demo():
    bh = BulkheadExecutor(max_concurrent=2)
    results = []

    async def send(tenant: str, req_id: int):
        try:
            resp = await bh.execute(
                tenant,
                mock_llm_call(tenant, req_id, latency=0.1),
            )
            results.append((tenant, req_id, "OK", resp))
        except BulkheadFull as e:
            results.append((tenant, req_id, "REJECTED", str(e)))

    # Tenant A floods with 5 concurrent requests; tenant B sends 2
    tasks = [
        send("tenant-A", i) for i in range(5)
    ] + [
        send("tenant-B", i) for i in range(2)
    ]
    await asyncio.gather(*tasks)

    print(f"{'Tenant':<12} {'Req':>4} {'Status':>10}  Detail")
    print("-" * 60)
    for tenant, req_id, status, detail in sorted(results):
        detail_short = detail[:50] if len(detail) > 50 else detail
        print(f"{tenant:<12} {req_id:>4} {status:>10}  {detail_short}")


await run_bulkhead_demo()

Tenant A's burst fills its two slots and the remaining three are rejected immediately, while both of tenant B's requests succeed unaffected. Isolation is complete: tenant A's failure budget does not touch tenant B.

:::{.callout-note}
In a real service the bulkhead limit per tenant would be configurable by tier — premium clients receive more concurrency slots than standard clients. A `TenantConfig` registry mapping tenant IDs to their contracted SLA tier is a natural extension.

:::

## Graceful Degradation

Graceful degradation is the commitment that a service will never return an unhandled exception to the caller. Instead, it returns the best answer it can given the current failure state, clearly annotated with its degradation level. We define a `DegradationLevel` enum with three values that correspond to increasingly conservative behavior:

- **FULL** — complete LLM analysis with structured reasoning and cited evidence.
- **PARTIAL** — classification verdict only; the reasoning step was skipped (e.g., due to a timeout).
- **SAFE_DEFAULT** — the service cannot classify at all; the request is flagged for mandatory human review.

The invariant is that `SAFE_DEFAULT` always has `verdict="REVIEW_REQUIRED"`. Downstream consumers can check `level` to decide whether to auto-approve (FULL, COMPLIANT) or route to a human queue (anything else).

Implementing the graceful reviewer:

In [ ]:
from dataclasses import dataclass as dc


class DegradationLevel(Enum):
    FULL         = "full"          # complete LLM analysis
    PARTIAL      = "partial"       # classification only, no explanation
    SAFE_DEFAULT = "safe_default"  # flagged for human review


@dataclass
class DegradedResult:
    verdict: str
    explanation: str
    level: DegradationLevel
    error: str | None = None


SAFE_DEFAULT_RESULT = DegradedResult(
    verdict="REVIEW_REQUIRED",
    explanation="Service unavailable. Routed to human review queue.",
    level=DegradationLevel.SAFE_DEFAULT,
)


class GracefulComplianceReviewer:
    """Three-tier graceful degradation for compliance review."""

    def __init__(self, full_client, partial_client):
        self._full    = full_client
        self._partial = partial_client

    def _full_review(self, text: str) -> DegradedResult:
        msgs = [
            {"role": "system", "content": (
                "You are a compliance officer. Reply with JSON: "
                '{"verdict": "COMPLIANT|NON_COMPLIANT", "reason": "<one sentence>"}'
            )},
            {"role": "user", "content": text},
        ]
        raw = self._full.complete(msgs)
        data = json.loads(raw)                               # <1>
        return DegradedResult(
            verdict=data["verdict"],
            explanation=data["reason"],
            level=DegradationLevel.FULL,
        )

    def _partial_review(self, text: str) -> DegradedResult:
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text},
        ]
        verdict = self._partial.complete(msgs).strip().upper()
        return DegradedResult(
            verdict=verdict,
            explanation="Partial review: classification only (explanation unavailable).",
            level=DegradationLevel.PARTIAL,
        )

    def review(self, text: str) -> DegradedResult:
        try:
            return self._full_review(text)                   # <2>
        except Exception as e1:
            print(f"  [Graceful] full review failed: {e1}")
            try:
                result = self._partial_review(text)          # <3>
                result.error = str(e1)
                return result
            except Exception as e2:
                print(f"  [Graceful] partial review failed: {e2}")
                r = DegradedResult(**vars(SAFE_DEFAULT_RESULT))
                r.error = str(e2)                            # <4>
                return r

1. `json.loads` will raise if the model produces malformed JSON — this is intentional. A parse failure escalates to the partial path rather than silently returning a corrupted result.
2. We always attempt the highest-quality path first.
3. The partial path uses a simpler prompt that is more robust — asking for a single word is harder to get wrong than structured JSON.
4. We preserve the error message on the result so the caller can log it for observability even though the degraded path succeeded.

Demonstrating all three tiers:

In [ ]:
class AlwaysFailLLM:
    model = "mock-broken"
    def complete(self, *a, **kw):
        raise RuntimeError("LLM service down")


TRADE_TEXT = "Execute margin call on account #4492 — positions exceed Reg T limit by 18%."

# Tier 1: both clients working (use real mini client for partial; stub JSON for full)
class StubFullLLM:
    model = "gpt-4o-stub"
    def complete(self, *a, **kw):
        return json.dumps({"verdict": "NON_COMPLIANT", "reason": "Position exceeds Reg T margin requirement."})

reviewer_full    = GracefulComplianceReviewer(StubFullLLM(), LLMClient("gpt-4o-mini"))
reviewer_partial = GracefulComplianceReviewer(AlwaysFailLLM(), LLMClient("gpt-4o-mini"))
reviewer_default = GracefulComplianceReviewer(AlwaysFailLLM(), AlwaysFailLLM())

for label, rev in [("FULL tier", reviewer_full), ("PARTIAL tier", reviewer_partial), ("SAFE_DEFAULT", reviewer_default)]:
    res = rev.review(TRADE_TEXT)
    print(f"\n--- {label} ---")
    print(f"  Verdict  : {res.verdict}")
    print(f"  Level    : {res.level.value}")
    print(f"  Explain  : {res.explanation}")

## Context Overflow Handling

LLM context windows are finite. GPT-4o supports 128k tokens; a naive implementation that concatenates all retrieved chunks and the user query can silently overflow if the retrieval system returns many long documents. When overflow occurs, the API raises an error (a permanent error — not retryable). The correct fix is **pre-call truncation**: measure the prompt size before sending and drop the lowest-ranked chunks until the payload fits.

We formalize the truncation policy. Let $C$ be the context limit, $S_{\text{sys}}$ the fixed token cost of the system prompt, $S_q$ the query tokens, and $\{s_1, s_2, \ldots, s_k\}$ the token sizes of retrieved chunks sorted by descending rank (rank 1 is most relevant). We greedily include chunks in rank order until adding the next chunk would overflow:

$$\text{included} = \left\{ i : \sum_{j=1}^{i} s_j \leq C - S_{\text{sys}} - S_q - \delta \right\}$$

where $\delta$ is a safety margin (we use $\delta = 200$ tokens) to accommodate the model's own output tokens and any prompt formatting overhead.

Implementing the guard:

In [ ]:
import warnings


def _approx_tokens(text: str) -> int:
    """Fast token approximation: ~4 characters per token (GPT-4 average)."""
    return max(1, len(text) // 4)


class ContextWindowGuard:
    """Truncates retrieved chunks to fit within the context window."""

    def __init__(
        self,
        context_limit: int = 128_000,
        output_reserve: int = 512,
        safety_margin: int = 200,
    ):
        self._limit   = context_limit
        self._reserve = output_reserve  # tokens reserved for model output
        self._margin  = safety_margin

    def fit(
        self,
        system_prompt: str,
        query: str,
        chunks: list[str],            # ordered by descending rank
    ) -> tuple[list[str], bool]:
        """Return (included_chunks, was_truncated)."""
        budget = (
            self._limit
            - _approx_tokens(system_prompt)   # <1>
            - _approx_tokens(query)
            - self._reserve
            - self._margin
        )

        included: list[str] = []
        used = 0
        for chunk in chunks:                  # <2>
            cost = _approx_tokens(chunk)
            if used + cost > budget:
                warnings.warn(
                    f"ContextWindowGuard: dropped {len(chunks) - len(included)} "
                    f"chunk(s) to fit within {self._limit}-token context window.",
                    stacklevel=2,
                )
                return included, True
            included.append(chunk)
            used += cost

        return included, False

1. We deduct fixed costs first: system prompt, query, output reserve, and safety margin. The remainder is the budget available for retrieved context.
2. Chunks are already sorted by descending relevance score, so we greedily include the best chunks and drop the tail. This is the minimal-information-loss truncation policy.

Testing the guard with a tight context limit:

In [ ]:
import warnings as _w
_w.simplefilter("always")

guard = ContextWindowGuard(
    context_limit=500,    # deliberately small to trigger truncation
    output_reserve=50,
    safety_margin=20,
)

system_prompt = "You are a compliance officer reviewing trade activity."
query = "Does this trade comply with Reg T margin requirements?"
chunks = [
    "Regulation T (12 CFR 220) requires that the initial margin on equity purchases be at least 50% of the purchase price. The broker-dealer must collect this margin within the settlement period.",   # rank 1
    "Federal Reserve Board Regulation T governs the amount of credit that broker-dealers may extend to customers for the purchase of securities. Margin calls must be satisfied within five business days.",  # rank 2
    "Maintenance margin requirements under FINRA Rule 4210 require customers to maintain at least 25% equity in their margin accounts at all times. Failure triggers a margin call.",                        # rank 3
    "Basel III capital requirements mandate that globally systemically important banks maintain additional capital buffers. This is unrelated to Reg T but is included as a low-ranked retrieval result.",   # rank 4 (low relevance)
]

included, truncated = guard.fit(system_prompt, query, chunks)
print(f"Included {len(included)}/{len(chunks)} chunks (truncated={truncated})")
for i, c in enumerate(included):
    print(f"  [{i+1}] {c[:80]}...")

:::{.callout-tip}
The character-per-token approximation (`len(text) // 4`) underestimates token counts for texts heavy with numbers, symbols, or non-ASCII characters — common in financial documents. For production use, replace `_approx_tokens` with `tiktoken.encoding_for_model(model).encode(text)` to get exact token counts at the cost of a small per-call overhead.

:::

## Full Demo: ResilientComplianceService

We assemble all six patterns into a single `ResilientComplianceService`. The request path is:

1. **Bulkhead** — enforce per-tenant concurrency limit.
2. **ContextWindowGuard** — truncate retrieved context to fit the window.
3. **CircuitBreaker** — reject calls to a failing primary model.
4. **RetryClient** — retry transient errors with jitter.
5. **FallbackChain** — cascade to cheaper models on failure.
6. **GracefulDegradation** — guarantee a safe response at every tier.

We then run a simulation with 20 requests from two tenants where the primary model fails probabilistically at rate $p = 0.4$.

Defining the probabilistic mock LLM for the simulation:

In [ ]:
class ProbabilisticLLM:
    """Stub LLM that fails with probability `fail_p` on each call."""

    def __init__(
        self,
        fail_p: float = 0.4,
        response: str = "COMPLIANT",
        error_type: type[Exception] = RateLimitError,
        seed: int = 42,
    ):
        self._fail_p     = fail_p
        self._response   = response
        self._error_type = error_type
        self._rng        = random.Random(seed)
        self.model       = "mock-probabilistic"

    def complete(self, messages, **kwargs):
        if self._rng.random() < self._fail_p:
            raise self._error_type("Simulated failure")
        return self._response

Assembling the resilient service:

In [ ]:
from dataclasses import dataclass, field


@dataclass
class ServiceResponse:
    tenant: str
    req_id: int
    verdict: str
    level: str
    source: str
    error: str | None = None


class ResilientComplianceService:
    """Compliance reviewer combining all six resilience patterns."""

    def __init__(
        self,
        primary_llm,
        fallback_llm,
        max_concurrent_per_tenant: int = 3,
        cb_window: int = 5,
        cb_threshold: float = 0.6,
        cb_cooldown: float = 2.0,
        context_limit: int = 128_000,
    ):
        self._guard    = ContextWindowGuard(context_limit=context_limit)       # <1>
        self._cb       = CircuitBreaker(                                        # <2>
            RetryClient(primary_llm, max_attempts=3, base=0.01, cap=0.1),
            window=cb_window, failure_threshold=cb_threshold, cooldown=cb_cooldown,
        )
        self._fallback = fallback_llm                                           # <3>
        self._bulkhead = BulkheadExecutor(max_concurrent=max_concurrent_per_tenant)

    def _do_review(self, text: str, chunks: list[str]) -> ServiceResponse:
        sys_prompt = "You are a compliance officer. Reply COMPLIANT or NON_COMPLIANT."
        kept, truncated = self._guard.fit(sys_prompt, text, chunks)            # <4>
        context = "\n".join(kept)
        full_text = f"{context}\n\nTrade: {text}" if context else text
        messages  = [{"role": "system", "content": sys_prompt},
                     {"role": "user",   "content": full_text}]
        try:
            verdict = self._cb.complete(messages).strip().upper()              # <5>
            return ServiceResponse("", 0, verdict, "full", "primary")
        except (CircuitBreakerOpen, RateLimitError, RuntimeError) as e:
            try:
                verdict = self._fallback.complete(messages).strip().upper()    # <6>
                return ServiceResponse("", 0, verdict, "partial", "fallback", str(e))
            except Exception as e2:
                return ServiceResponse("", 0, "REVIEW_REQUIRED", "safe_default", "none", str(e2))

    async def review(self, tenant: str, req_id: int, text: str,
                     chunks: list[str] | None = None) -> ServiceResponse:
        chunks = chunks or []
        try:
            result = await self._bulkhead.execute(                             # <7>
                tenant,
                asyncio.to_thread(self._do_review, text, chunks),
            )
            result.tenant = tenant
            result.req_id = req_id
            return result
        except BulkheadFull as e:
            return ServiceResponse(tenant, req_id, "REVIEW_REQUIRED", "safe_default", "none", str(e))

1. The context guard runs first, before any network call, so we never send an oversized prompt.
2. The circuit breaker wraps a `RetryClient`, so transient errors are retried with jitter before the breaker counts them as failures.
3. The fallback LLM is accessed directly — it has no circuit breaker — since it is already the last-resort before the safe default.
4. Truncation is logged via `warnings.warn`; the `truncated` flag can be used for metrics.
5. Errors from the primary path (circuit open, retries exhausted, or any runtime error) are caught and escalated to the fallback.
6. If the fallback also fails, we return `SAFE_DEFAULT` — guaranteeing the degraded-safe rate SLO.
7. The bulkhead wraps the entire synchronous `_do_review` call, including retries, so the semaphore slot is held for the full duration of the request.

Running the simulation:

In [ ]:
SAMPLE_TRADES = [
    "Sell 10,000 shares of ACME Corp for hedge fund client #221. Client is on restricted list.",
    "Execute market order: buy 500 SPY contracts for proprietary desk account #881.",
    "Transfer $2.1M from omnibus account to client #334. No AML flag in system.",
    "Margin call: account #992 is below maintenance margin. Liquidate 15% of position.",
    "Block trade: buy 50,000 shares of ZYX Inc at $42.10 for pension fund client #17.",
]

SAMPLE_CONTEXT = [
    "Reg T requires 50% initial margin on equity positions.",
    "FINRA Rule 4110 governs maintenance margin requirements.",
    "AML policy requires enhanced due diligence for transfers over $1M.",
]


async def run_simulation(n_requests: int = 20, fail_p: float = 0.4):
    primary  = ProbabilisticLLM(fail_p=fail_p, response="COMPLIANT", seed=7)
    fallback = ProbabilisticLLM(fail_p=0.1,    response="COMPLIANT", seed=13)

    svc = ResilientComplianceService(
        primary_llm=primary,
        fallback_llm=fallback,
        max_concurrent_per_tenant=3,
        cb_window=4,
        cb_threshold=0.75,
        cb_cooldown=0.05,
        context_limit=100_000,
    )

    tenants = ["compliance-team", "trade-desk"]
    tasks = []
    rng = random.Random(99)
    for i in range(n_requests):
        tenant = rng.choice(tenants)
        trade  = rng.choice(SAMPLE_TRADES)
        tasks.append(svc.review(tenant, i, trade, SAMPLE_CONTEXT))

    responses: list[ServiceResponse] = await asyncio.gather(*tasks)

    counts = {"full": 0, "partial": 0, "safe_default": 0}
    for r in responses:
        counts[r.level] += 1

    total = len(responses)
    print(f"Simulation: {total} requests, primary fail_p={fail_p}")
    print(f"{'Level':<16} {'Count':>6} {'Rate':>8}")
    print("-" * 32)
    for level, count in counts.items():
        print(f"{level:<16} {count:>6} {count/total:>8.1%}")
    print()
    print("Sample responses:")
    for r in responses[:5]:
        err_str = f" (err: {r.error[:40]}...)" if r.error else ""
        print(f"  [{r.tenant}] req={r.req_id:2d} verdict={r.verdict:<17} level={r.level} source={r.source}{err_str}")


await run_simulation(n_requests=20, fail_p=0.4)

Despite a 40% primary failure rate, the service delivers actionable results on every request: some at full quality (primary succeeded or retried successfully), some at partial quality (fallback), and the remainder at safe default (flagged for human review). The degraded-safe rate SLO — that no request returns an unhandled exception — is met unconditionally.

:::{.callout-important}
The resilience patterns here are defense-in-depth, not a substitute for model quality. A service that routinely falls back to safe default is not resilient — it is broken. Monitor the ratio of full-to-degraded responses as a leading indicator of model health, and page on-call when the fallback rate exceeds 5% over a 5-minute window.

:::

## Exercises

1. **Add observability to `CircuitBreaker`.** Extend `CircuitBreaker` with a `metrics()` method returning `{"state": ..., "failure_rate": ..., "total_calls": ..., "total_rejections": ...}`. Instrument `ResilientComplianceService` to call `metrics()` after each batch of requests and print a summary table.

2. **Implement adaptive bulkhead sizing.** Modify `BulkheadExecutor` to dynamically adjust `max_concurrent` per tenant based on observed p99 latency: if latency exceeds 1.5s, halve the limit; if latency is below 0.5s for 30 consecutive requests, increase by 1 (up to a configured maximum). Use a rolling latency window of 20 samples.

3. **Integrate `ContextWindowGuard` with tiktoken.** Replace `_approx_tokens` with an exact token counter using `tiktoken.encoding_for_model("gpt-4o")`. Measure the approximation error on the `SAMPLE_CONTEXT` strings above. At what document length does the approximation first disagree with the exact count by more than 10%?

---

$\blacksquare$